<a href="https://colab.research.google.com/github/zyuzyunda/helper/blob/main/clean_skills_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import re
import pandas as pd
import numpy as np
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_distances

In [5]:
def clean_text(text):
    """Приведение текста к нижнему регистру и удаление спецсимволов"""
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Zа-яА-Я0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def embed_texts(texts, model_name="paraphrase-multilingual-MiniLM-L12-v2"):
    """Генерация эмбеддингов с помощью SentenceTransformer"""
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    return embeddings

def reduce_dimensionality(embeddings, n_neighbors=15, min_dist=0.1, metric="cosine"):
    umap_model = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, metric=metric, random_state=42)
    umap_embeddings = umap_model.fit_transform(embeddings)
    return umap_embeddings

def cluster_embeddings(umap_embeddings, min_cluster_size=5):
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean")
    labels = clusterer.fit_predict(umap_embeddings)
    return labels

def assign_cluster_names(df, embeddings, name_column):
    """Для каждого кластера выбираем самое репрезентативное название"""
    cluster_names = {}
    for cluster_id in df['cluster'].unique():
        if cluster_id == -1:
            continue  # пропускаем шум
        idxs = np.where(df['cluster'] == cluster_id)[0]
        cluster_embs = embeddings[idxs]
        center = cluster_embs.mean(axis=0, keepdims=True)
        distances = cosine_distances(cluster_embs, center).flatten()
        best_idx = idxs[np.argmin(distances)]
        cluster_names[cluster_id] = df.loc[best_idx, name_column]

    cluster_names_df = pd.DataFrame(list(cluster_names.items()), columns=['cluster', 'cluster_name'])
    df_with_names = df.merge(cluster_names_df, on='cluster', how='left')
    df_with_names['cluster_name'] = df_with_names.apply(
        lambda row: row[name_column] if row['cluster'] == -1 else row['cluster_name'],
        axis=1
    )
    return df_with_names[['cluster_name', 'cluster'] + [col for col in df_with_names.columns if col not in ['cluster_name','cluster']]]

def save_results(df_with_names, output_clusters_file, output_df_file):
    df_with_names.to_csv(output_df_file, index=False)
    df_with_names[['cluster','cluster_name']].drop_duplicates().to_csv(output_clusters_file, index=False)

def process_file_dynamic(input_file, output_df_file, output_clusters_file):
    base_name = os.path.splitext(os.path.basename(input_file))[0]
    df = pd.read_csv(input_file)
    df = df.rename(columns={df.columns[0]: f'{base_name}_id', df.columns[1]: f'{base_name}'})
    df['clean'] = df[f'{base_name}'].apply(clean_text)
    embeddings = embed_texts(df['clean'].tolist())
    umap_embeddings = reduce_dimensionality(embeddings)
    df['cluster'] = cluster_embeddings(umap_embeddings)
    df_final = assign_cluster_names(df, embeddings, name_column=f'{base_name}')
    save_results(df_final, output_clusters_file, output_df_file)
    print(f"Обработан файл: {input_file}, всего навыков: {len(df_final)}")
    return df_final

if __name__ == "__main__":
    DATA_PATH = '/content/drive/MyDrive/ИТМО/Хакатон 1/data'

    files_to_process = [
        'competencies_rows.csv',
        'tools_rows.csv',
        'soft_skills_rows.csv'
    ]

    for file_name in files_to_process:
        input_file = f"{DATA_PATH}/{file_name}"
        output_df = f"{DATA_PATH}/{file_name.replace('.csv','')}_clustered_cleaned.csv"
        output_clusters = f"{DATA_PATH}/{file_name.replace('.csv','')}_clusters.csv"
        df_final = process_file_dynamic(input_file, output_df, output_clusters)
        print(df_final.head(5))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Обработан файл: /content/drive/MyDrive/ИТМО/Хакатон 1/data/tools_rows.csv, всего навыков: 1927
  cluster_name  cluster  tools_rows_id tools_rows   clean
0          NaN       -1              1        NaN     nan
1     NetworkX       69              2       .NET     net
2        OL8/9       58              3     11Labs  11labs
3      Xgboost       86              4      199-И   199 и
4           1С       -1              5         1С      1с


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Обработан файл: /content/drive/MyDrive/ИТМО/Хакатон 1/data/soft_skills_rows.csv, всего навыков: 460
                cluster_name  cluster  soft_skills_rows_id  \
0        владение английским        0                    1   
1  Аналитические способности       20                    2   
2       Системность мышления       11                    3   
3  Аналитические способности       20                    4   
4   Внимательность к деталям        3                    5   

       soft_skills_rows                 clean  
0      Advanced English      advanced english  
1  Analytical abilities  analytical abilities  
2    Analytical mindset    analytical mindset  
3     Analytical skills     analytical skills  
4   Attention to detail   attention to detail  


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
